<a href="https://colab.research.google.com/github/rmadatt/ADLAB/blob/main/Bizkits.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install OpenAI, Supabase client, and PyPDF for document extraction
!pip install -q openai supabase pypdf python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 18.7 MB/s eta 0:00:00


In [2]:
import os
from openai import OpenAI
from supabase import create_client, Client

# Replace these with your actual keys
OPENAI_API_KEY = "your-openai-api-key"
SUPABASE_URL = "https://your-supabase-project.supabase.co"
SUPABASE_KEY = "your-supabase-service-role-key"

client = OpenAI(api_key=OPENAI_API_KEY)
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

print("Connections initialized successfully!")

Connections initialized successfully!


In [3]:
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path: str) -> str:
    """Extracts raw text from a business PDF (e.g., menu or service list)."""
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text

def add_business_data(business_name: str, contact_info: str, raw_text: str):
    """Generates an embedding and saves business data into Supabase."""

    # Combined content for better vector search results
    full_content = f"Business Name: {business_name}\nContact: {contact_info}\nDetails: {raw_text}"

    # 1. Generate Embedding
    response = client.embeddings.create(
        input=full_content,
        model="text-embedding-3-small"
    )
    embedding = response.data[0].embedding

    # 2. Insert into Supabase
    data = {
        "business_name": business_name,
        "contact_info": contact_info,
        "raw_text": raw_text,
        "embedding": embedding
    }

    res = supabase.table("businesses").insert(data).execute()
    print(f"Successfully added {business_name} to search database!")
    return res

# --- Example Usage in Colab ---
# Add a sample business
add_business_data(
    business_name="Mario's Late Night Plumbing",
    contact_info="555-0199 | mario@plumbing.com | Open 24/7",
    raw_text="We offer emergency pipe repair, water heater fixing, and clogged drain clearing after hours."
)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: your-ope*******-key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [4]:
def search_businesses(user_query: str):
    """Searches vector database for matching businesses and generates an answer."""

    # 1. Generate Embedding for User Query
    query_embedding_res = client.embeddings.create(
        input=user_query,
        model="text-embedding-3-small"
    )
    query_embedding = query_embedding_res.data[0].embedding

    # 2. Match Embeddings in Supabase (calls a stored SQL procedure)
    rpc_response = supabase.rpc(
        "match_businesses",
        {
            "query_embedding": query_embedding,
            "match_threshold": 0.3,
            "match_count": 3
        }
    ).execute()

    matches = rpc_response.data

    if not matches:
        return "No matching small businesses found for your request."

    # 3. Format Context for LLM Synthesis
    context = "\n---\n".join([
        f"Business: {m['business_name']}\nContact: {m['contact_info']}\nDetails: {m['raw_text']}"
        for m in matches
    ])

    system_prompt = (
        "You are a helpful local business search assistant. "
        "Recommend the best matching business(es) based on the context below. "
        "Always provide their contact details and explain clearly why they match the user's request."
    )

    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"User Query: {user_query}\n\nAvailable Businesses:\n{context}"}
        ]
    )

    return completion.choices[0].message.content

# --- Test the Search ---
user_question = "Who can I call to fix a burst water pipe right now in the evening?"
answer = search_businesses(user_question)
print("=== AI Search Result ===")
print(answer)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: your-ope*******-key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [5]:
!pip install -q gradio


In [ ]:
import gradio as gr
from pypdf import PdfReader

# Helper function to extract text directly from a Gradio file object
def process_uploaded_pdf(file_obj) -> str:
    """Extracts text from an uploaded PDF file path."""
    if file_obj is None:
        return ""

    try:
        reader = PdfReader(file_obj.name)
        extracted_text = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                extracted_text += text + "\n"
        return extracted_text.strip()
    except Exception as e:
        print(f"Error reading PDF: {e}")
        return ""

# Updated Wrapper for Adding Business Data with PDF Support
def gradio_add_business_with_pdf(name, contact, text_details, pdf_file):
    if not name or not contact:
        return "⚠️ Business Name and Contact Info are required."

    # 1. Extract text from PDF if one was uploaded
    pdf_text = process_uploaded_pdf(pdf_file)

    # 2. Combine text input and PDF content
    combined_details = ""
    if text_details.strip():
        combined_details += f"Manual Notes:\n{text_details.strip()}\n\n"
    if pdf_text:
        combined_details += f"Document Content (PDF):\n{pdf_text}"

    if not combined_details.strip():
        return "⚠️ Please provide either text details or upload a valid PDF."

    # 3. Call vector embedding function
    try:
        add_business_data(
            business_name=name,
            contact_info=contact,
            raw_text=combined_details
        )
        status_msg = f"✅ Successfully embedded '{name}'!"
        if pdf_text:
            status_msg += " (PDF text extracted and indexed)"
        return status_msg
    except Exception as e:
        return f"❌ Error saving business: {str(e)}"

# --- Construct the Updated Gradio Interface ---
with gr.Blocks(theme=gr.themes.Soft(), title="Local Business AI Search") as demo:
    gr.Markdown("# 🔍 Small Business Search AI")
    gr.Markdown("Search local business data or upload menus and price lists directly as PDFs.")

    with gr.Tabs():
        # TAB 1: User Search Interface
        with gr.TabItem("User Search"):
            search_input = gr.Textbox(
                label="Search Query",
                placeholder="e.g., Which bakery offers gluten-free custom birthday cakes?",
                lines=2
            )
            search_button = gr.Button("Search", variant="primary")
            search_output = gr.Markdown(label="AI Search Results")

            search_button.click(
                fn=gradio_search,
                inputs=search_input,
                outputs=search_output
            )

        # TAB 2: Business Onboarding Portal (With PDF Support)
        with gr.TabItem("Add Business Data"):
            gr.Markdown("### Register Business & Upload Service Guides / Menus")

            with gr.Row():
                name_input = gr.Textbox(label="Business Name*", placeholder="e.g., Sunshine Bakery")
                contact_input = gr.Textbox(label="Contact Info*", placeholder="e.g., 555-0199 | orders@sunshine.com")

            details_input = gr.Textbox(
                label="Manual Service Details / Description (Optional)",
                placeholder="Add general notes, opening hours, or quick pricing details...",
                lines=3
            )

            # PDF File Component
            pdf_input = gr.File(
                label="Upload Menu or Service List (PDF)",
                file_types=[".pdf"],
                type="filepath"
            )

            add_button = gr.Button("Embed & Save to Database", variant="secondary")
            add_output = gr.Label(label="Status")

            add_button.click(
                fn=gradio_add_business_with_pdf,
                inputs=[name_input, contact_input, details_input, pdf_input],
                outputs=add_output
            )

# Launch in Colab with shareable link
demo.launch(share=True, debug=True)

/tmp/ipykernel_1516/4065606809.py:55: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Local Business AI Search") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://26939a19fe596ef58e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Run this in Colab to install the chunking library:
# !pip install -q langchain-text-splitters

from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_text(text: str, chunk_size: int = 500, chunk_overlap: int = 50):
    """Splits long PDF text into smaller overlapping chunks for precise vector search."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    return text_splitter.split_text(text)

def add_business_data_chunked(business_name: str, contact_info: str, raw_text: str):
    """Chunks input text and creates an embedding entry for each chunk."""

    # 1. Chunk the raw text
    chunks = chunk_text(raw_text)
    print(f"Split document into {len(chunks)} chunks for {business_name}.")

    inserted_records = []

    # 2. Embed each chunk separately
    for i, chunk in enumerate(chunks):
        full_chunk_text = f"Business Name: {business_name}\nContact: {contact_info}\nChunk {i+1}:\n{chunk}"

        response = client.embeddings.create(
            input=full_chunk_text,
            model="text-embedding-3-small"
        )
        embedding = response.data[0].embedding

        record = {
            "business_name": business_name,
            "contact_info": contact_info,
            "raw_text": chunk, # Save individual chunk text
            "embedding": embedding
        }
        inserted_records.append(record)

    # 3. Batch insert all chunk vectors into Supabase
    res = supabase.table("businesses").insert(inserted_records).execute()
    return res